**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [2]:
import os
os.environ["DHCORE_LOG_LEVEL"]="DEBUG"
import digitalhub as dh
dh.refresh_token()

2026-09-18 11:21:51,331 - dhcore.digitalhub.stores.client.auth.refresh - DEBUG - Starting credential refresh attempt 1.
2026-09-18 11:21:51,332 - dhcore.digitalhub.stores.client.auth.refresh - DEBUG - Starting credential refresh with auth type 'oauth2'.
2026-09-18 11:21:51,338 - dhcore.digitalhub.stores.client.auth.refresh - DEBUG - Request: HTTP GET http://rsde-platform-core.rsde-platform.svc.cluster.local:8080/.well-known/openid-configuration - Status: 200 - Headers: {'User-Agent': 'python-requests/2.34.2', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'} - Request body: None - Response body: {"response_types_supported":["code"],"jwks_uri":"https://core.rsde.atlas.fbk.eu/auth/jwks","grant_types_supported":["refresh_token","client_credentials","authorization_code","urn:ietf:params:oauth:grant-type:token-exchange"],"token_endpoint_auth_methods_supported":["client_secret_post","none","client_secret_basic"],"scopes_supported":["credentials","openid","offli

In [1]:
import digitalhub as dh

In [3]:
dh.refresh_token()

HTTPError: 400 Client Error: Bad Request for url: https://core.rsde.atlas.fbk.eu/auth/token

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

profiles = dh.get_k8s_resource_profiles()
print(profiles)

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

In [ ]:
help(project.log_artifact)

**SETUP PARAMETERS**

In [ ]:
# Parametri Job   
job_name = "test_encoders_visual_v7"                                
dataset = "Test" 
test_sar = False
test_opt = True                                       
#handler = pretrain_encoders                                         

parametri = {     
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                      
    "n_images2": 4, "n_channels2": 10,  
    "output_dim": 512,                              
    "mamba": False, 
    "dataset": dataset,
    "test_sar": test_sar,                                 
    "test_opt": test_opt,
    "weights_s1": "encoder-s1-weights_train_s1_v3_Standard_200",
    "weights_s2": "encoder-s2-weights_train_s2_v3_Standard_200",
    "n_samples": 10,         
    "recon_channels": [2,1,0]
}                                                              

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
test_train_func = project.new_function(
    name= f'encoders-Floods_{dataset}_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="test_encoders_visual", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30","digitalhub==0.16.0b3", "digitalhub-runtime-python==0.16.0b3", "torch==2.1.2", "matplotlib==3.10.9"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = test_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_test_encoders = test_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run test_encoders avviato: {run_test_encoders.id}")
print(run_test_encoders.status.state)
print(run_test_encoders.status.message)